# IMRPhenomXPHM / SpinTaylor fitting factor against NRSur7dq4

This is a workflow template for expensive model comparisons. It keeps waveform generation behind user-supplied hooks, then shows how to run three fitting-factor searches:

1. optimize all intrinsic candidate parameters,
2. optimize intrinsic parameters plus full 3D Wigner rotations,
3. optimize only spin angles and phase-like parameters for a fixed intrinsic configuration.

The cells are intentionally output-free in the repository. Real waveform generation requires a local LALSuite/surrogate setup, and the expensive searches are left commented until you are ready to run them.

In [ ]:
from __future__ import annotations

import importlib
from pprint import pprint

import numpy as np

import waveformtools.comparison  # installs ModesArray.fitting_factor
from waveformtools.comparison import AlignmentSpec, FittingFactorConfig, ModeComparisonConfig, RotationSpec


def optional_import(name):
    try:
        return importlib.import_module(name)
    except ImportError as exc:
        raise RuntimeError(f"Install or activate the dependency needed for {name!r} before running this notebook.") from exc


# Uncomment when running real waveform generation.
# lal = optional_import("lal")
# lalsimulation = optional_import("lalsimulation")
# gwsurrogate = optional_import("gwsurrogate")


## Waveform generation hooks

Both hooks return `waveformtools.modes_array.ModesArray` objects. The candidate hook accepts arbitrary user-chosen optimizer parameters; this cell converts the convenience parameters it knows about and passes LAL-style parameters to `LALWaveformModel`.

In [ ]:
BASE_WAVEFORM_PARAMETERS = {
    "mass1": 40.0,
    "mass2": 35.0,
    "spin1x": 0.08,
    "spin1y": -0.03,
    "spin1z": 0.25,
    "spin2x": -0.04,
    "spin2y": 0.02,
    "spin2z": -0.15,
    "distance": 400.0,
    "inclination": 0.7,
    "phi_ref": 0.2,
    "f_lower": 20.0,
    "f_ref": 20.0,
    "f_max": 512.0,
    "delta_t": 1.0 / 2048.0,
    "delta_f": 1.0 / 4.0,
    "ell_max": 4,
}

LAL_PARAMETER_KEYS = set(BASE_WAVEFORM_PARAMETERS) | {
    "approximant",
    "PhenomXHMReleaseVersion",
    "PhenomXPrecVersion",
    "debug",
    "lvcnr_file_path",
}


def _masses_from_total_mass_and_q(total_mass, q):
    mass2 = total_mass / (1.0 + q)
    return q * mass2, mass2


def _spin_components(magnitude, theta, phi):
    return (
        magnitude * np.sin(theta) * np.cos(phi),
        magnitude * np.sin(theta) * np.sin(phi),
        magnitude * np.cos(theta),
    )


def resolve_lal_parameters(approximant, **user_parameters):
    """Convert user optimizer parameters into LALWaveformModel parameters."""
    parameters = dict(BASE_WAVEFORM_PARAMETERS)
    parameters.update(user_parameters)
    parameters["approximant"] = approximant

    total_mass = parameters.pop(
        "total_mass",
        parameters.get("mass1", 0.0) + parameters.get("mass2", 0.0),
    )
    q = parameters.pop("q", None)
    if q is not None:
        parameters["mass1"], parameters["mass2"] = _masses_from_total_mass_and_q(total_mass, q)

    for index in (1, 2):
        for component in ("x", "y", "z"):
            chi_key = f"chi{index}{component}"
            spin_key = f"spin{index}{component}"
            if chi_key in parameters:
                parameters[spin_key] = parameters.pop(chi_key)
        chi_magnitude_key = f"chi{index}_magnitude"
        spin_magnitude_key = f"spin{index}_magnitude"
        if chi_magnitude_key in parameters and spin_magnitude_key not in parameters:
            parameters[spin_magnitude_key] = parameters.pop(chi_magnitude_key)

    for index in (1, 2):
        magnitude = parameters.pop(f"spin{index}_magnitude", None)
        theta = parameters.pop(f"spin{index}_theta", None)
        phi = parameters.pop(f"spin{index}_phi", None)
        if magnitude is not None or theta is not None or phi is not None:
            if magnitude is None or theta is None or phi is None:
                raise ValueError(f"spin{index}_magnitude/theta/phi must be supplied together")
            sx, sy, sz = _spin_components(magnitude, theta, phi)
            parameters[f"spin{index}x"] = sx
            parameters[f"spin{index}y"] = sy
            parameters[f"spin{index}z"] = sz

    if "reference_phase" in parameters:
        parameters["phi_ref"] = parameters.pop("reference_phase")

    lal_parameters = {key: value for key, value in parameters.items() if key in LAL_PARAMETER_KEYS}
    ignored = sorted(set(parameters) - set(lal_parameters))
    return lal_parameters, ignored


def generate_lal_modes(approximant=None, **user_parameters):
    optional_import("lal")
    optional_import("lalsimulation")
    from waveformtools.models.lal import LALWaveformModel

    approximant = approximant or user_parameters.pop("approximant", None)
    if approximant is None:
        raise ValueError("Pass an approximant argument or include it in fixed_parameters.")
    user_parameters.pop("approximant", None)
    parameters, ignored = resolve_lal_parameters(approximant, **user_parameters)
    if ignored:
        print("Ignoring parameters not consumed by the LAL hook:", ignored)
    model = LALWaveformModel(parameters_dict=parameters)
    return model.get_td_waveform_modes(dimensionless=True)


def load_nrsur7dq4_reference(**nr_parameters):
    return generate_lal_modes("NRSur7dq4", **nr_parameters)


def generate_phenomxphm_spintaylor_candidate(**candidate_parameters):
    return generate_lal_modes("IMRPhenomXPHM", **candidate_parameters)


def print_fitting_result(label, result):
    print(f"\n{label}")
    print("match:", result.match)
    print("mismatch:", result.mismatch)
    print("optimizer:", result.optimizer)
    print("waveform generations:", result.n_waveform_generations)
    print("best candidate/generator parameters:")
    pprint(result.best_parameters.get("generator", {}))
    print("best alignment/rotation parameters:")
    pprint(result.best_parameters.get("alignment", {}))


## Reference and baseline settings

In [ ]:
nr_reference_parameters = {
    "q": 2.0,
    "chi1x": 0.1,
    "chi1y": 0.0,
    "chi1z": 0.3,
    "chi2x": -0.05,
    "chi2y": 0.02,
    "chi2z": -0.2,
}

target_modes = load_nrsur7dq4_reference(**nr_reference_parameters)

comparison_base = ModeComparisonConfig(
    ell_min=2,
    ell_max=4,
    alignment=AlignmentSpec(
        time_alignment="peak_total_news_power",
        time_domain_policy="resample_to_reference",
        phase_alignment="orbital_phase_and_global",
        resample_method="linear",
        optimize_time_shift=True,
    ),
)


## Search 1: optimize all intrinsic candidate parameters

In [ ]:
all_intrinsic_config = FittingFactorConfig(
    comparison=comparison_base,
    variable_parameters={
        "q": (1.0, 6.0),
        "chi1x": (-0.8, 0.8),
        "chi1y": (-0.8, 0.8),
        "chi1z": (-0.8, 0.8),
        "chi2x": (-0.8, 0.8),
        "chi2y": (-0.8, 0.8),
        "chi2z": (-0.8, 0.8),
    },
    initial_parameters=nr_reference_parameters,
    fixed_parameters={"approximant": "IMRPhenomXPHM"},
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_all_intrinsic = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=all_intrinsic_config,
# )
# print_fitting_result("All intrinsic parameters", result_all_intrinsic)


## Search 2: optimize intrinsic parameters plus full 3D Wigner rotation

In [ ]:
intrinsic_plus_rotation_config = FittingFactorConfig(
    comparison=ModeComparisonConfig(
        ell_min=2,
        ell_max=4,
        alignment=comparison_base.alignment,
        rotation=RotationSpec(
            kind="wigner",
            optimize_parameters=("alpha", "beta", "gamma"),
            parameter_bounds={
                "alpha": (-np.pi, np.pi),
                "beta": (-np.pi, np.pi),
                "gamma": (-np.pi, np.pi),
            },
        ),
    ),
    variable_parameters=all_intrinsic_config.variable_parameters,
    initial_parameters=all_intrinsic_config.initial_parameters,
    fixed_parameters=all_intrinsic_config.fixed_parameters,
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_intrinsic_plus_rotation = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=intrinsic_plus_rotation_config,
# )
# print_fitting_result("Intrinsic plus full 3D Wigner rotation", result_intrinsic_plus_rotation)


## Search 3: fixed intrinsic parameters, optimize spin angles and phase-like parameters

In [ ]:
fixed_intrinsic_parameters = {
    "q": nr_reference_parameters["q"],
    "spin1_magnitude": 0.35,
    "spin2_magnitude": 0.25,
    "approximant": "IMRPhenomXPHM",
}

spin_angles_and_phase_config = FittingFactorConfig(
    comparison=ModeComparisonConfig(
        ell_min=2,
        ell_max=4,
        alignment=comparison_base.alignment,
        rotation=RotationSpec(kind="z_axis", optimize_angle=True, angle_bounds=(-np.pi, np.pi)),
    ),
    variable_parameters={
        "spin1_theta": (0.0, np.pi),
        "spin1_phi": (-np.pi, np.pi),
        "spin2_theta": (0.0, np.pi),
        "spin2_phi": (-np.pi, np.pi),
        "reference_phase": (-np.pi, np.pi),
    },
    initial_parameters={
        "spin1_theta": 0.7,
        "spin1_phi": 0.0,
        "spin2_theta": 1.0,
        "spin2_phi": 0.0,
        "reference_phase": 0.0,
    },
    fixed_parameters=fixed_intrinsic_parameters,
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_spin_angles_phase = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=spin_angles_and_phase_config,
# )
# print_fitting_result("Spin angles and phases", result_spin_angles_phase)
